In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import textwrap
import os

OUT_DIR = "eda_images"
os.makedirs(OUT_DIR, exist_ok=True)
df = pd.read_csv("/content/drive/MyDrive/StackOverFlow/survey_cleaned_base.csv")

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.titlesize': 14
})

def wrap_index(series, width=35):
    series.index = [textwrap.fill(str(idx), width) for idx in series.index]
    return series

def explode_multi(series, sep=";"):
    return series.dropna().str.split(sep).explode().str.strip().value_counts()

COUNTRY_SHORT = {
    "United States of America": "USA",
    "United Kingdom of Great Britain and Northern Ireland": "UK",
    "Russian Federation": "Russia",
    "Republic of Korea": "South Korea"
}

if "Country" in df.columns:
    country_counts = df["Country"].value_counts()
    country_counts.index = [COUNTRY_SHORT.get(c, c) for c in country_counts.index]
    fig, ax = plt.subplots(figsize=(10, 6))
    wrap_index(country_counts.head(20)).plot.barh(ax=ax, color="#5B9BD5")
    ax.set_xlabel("Количество респондентов")
    ax.set_title("Топ-20 стран по количество респондентов")
    ax.invert_yaxis()
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, "02_top20_countries.png"), dpi=300)
    plt.close(fig)

for col in ["MainBranch", "EdLevel"]:
    if col in df.columns:
        vc = df[col].value_counts()
        fig, ax = plt.subplots(figsize=(9, 5))
        wrap_index(vc).plot.barh(ax=ax, color="#5B9BD5")
        ax.invert_yaxis()
        ax.set_title(col)
        fig.tight_layout()
        fig.savefig(os.path.join(OUT_DIR, f"07_{col}.png"), dpi=300)
        plt.close(fig)

if "DevType" in df.columns:
    dt = explode_multi(df["DevType"])
    fig, ax = plt.subplots(figsize=(10, 6))
    wrap_index(dt.head(15), width=40).plot.barh(ax=ax, color="#5B9BD5")
    ax.invert_yaxis()
    ax.set_title("Типы разработчиков (топ-15)")
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, "06_devtype_top15.png"), dpi=300)
    plt.close(fig)

for col in ["YearsCode", "YearsCodePro"]:
    if col in df.columns:
        s = pd.to_numeric(df[col], errors="coerce")
        fig, ax = plt.subplots(figsize=(8, 4))
        s.dropna().clip(upper=50).hist(bins=50, ax=ax, color="#5B9BD5", edgecolor="white")
        ax.set_xlabel("Лет")
        ax.set_title(f"Распределение {col}")
        fig.tight_layout()
        fig.savefig(os.path.join(OUT_DIR, f"03_{col}_hist.png"), dpi=300)
        plt.close(fig)

if "Age" in df.columns and "MainBranch" in df.columns:
    so_age_order = ["Under 18 years old", "18-24 years old", "25-34 years old", "35-44 years old", "45-54 years old", "55-64 years old", "65 years or older"]
    prof_mask = df["MainBranch"] == "I am a developer by profession"
    prof_counts = df[prof_mask]["Age"].value_counts().reindex(so_age_order).fillna(0).astype(int)
    prof_pct_so = (prof_counts / prof_counts.sum() * 100).values

    def so_to_rosstat7(p):
        u18, a1824, a2534, a3544, a4554, a5564, a65p = p
        return np.array([u18 + a1824 * (2/7), a1824 * (5/7) + a2534 * 0.5, a2534 * 0.5 + a3544 * 0.5, a3544 * 0.5 + a4554 * 0.5, a4554 * 0.5 + a5564 * 0.5, a5564 * 0.5 + a65p * 0.6, a65p * 0.4])

    dev_pct = so_to_rosstat7(prof_pct_so)
    dev_pct = dev_pct / dev_pct.sum() * 100

    age_labels_rs = ["15-19", "20-29", "30-39", "40-49", "50-59", "60-69", "70+"]
    info_pct = np.array([0.5, 23.0, 37.8, 21.8, 12.4, 4.4, 0.2])
    edu_pct = np.array([0.1, 11.3, 26.2, 27.4, 24.8, 9.6, 0.6])
    health_pct = np.array([0.2, 11.2, 24.5, 27.8, 26.2, 9.6, 0.5])

    BG, FG = "#FFFFFF", "#1C1C1C"
    RED, GRAY1, GRAY2 = "#D9534F", "#BFBFBF", "#595959"

    fig = plt.figure(figsize=(14, 8), facecolor=BG)
    gs = fig.add_gridspec(1, 3, width_ratios=[10, 1.3, 10], wspace=0.0, left=0.03, right=0.97, top=0.90, bottom=0.28)
    axL, axC, axR = fig.add_subplot(gs[0], facecolor=BG), fig.add_subplot(gs[1], facecolor=BG), fig.add_subplot(gs[2], facecolor=BG)

    yy = np.arange(len(age_labels_rs))
    axL.barh(yy, info_pct, height=0.78, color=GRAY1, alpha=0.95)
    axL.barh(yy, dev_pct, height=0.42, color=RED, alpha=1.0)
    axL.invert_xaxis()
    axL.set_xlim(left=max(info_pct.max(), dev_pct.max()) * 1.08, right=0)

    axR.barh(yy, edu_pct, height=0.78, color=GRAY1, alpha=0.95)
    axR.barh(yy, health_pct, height=0.42, color=GRAY2, alpha=1.0)
    axR.set_xlim(left=0, right=max(edu_pct.max(), health_pct.max()) * 1.08)

    axC.set_xlim(0, 1); axC.set_ylim(yy.min()-0.6, yy.max()+0.6)
    axC.set_xticks([]); axC.set_yticks([])
    for sp in axC.spines.values(): sp.set_visible(False)
    for yi, lbl in zip(yy, age_labels_rs):
        axC.text(0.5, yi, lbl, ha="center", va="center", color=FG, fontsize=11, fontweight="bold")
    axC.text(0.5, yy.max()+0.35, "возраст,\nлет", ha="center", va="bottom", color=FG, style="italic")

    for ax in (axL, axR):
        ax.set_ylim(yy.min()-0.6, yy.max()+0.6)
        ax.set_yticks([])
        for side in ("top", "right", "left"): ax.spines[side].set_visible(False)
        ax.spines["bottom"].set_color(FG)
        ax.tick_params(axis="x", colors=FG)
        _lim = max(abs(ax.get_xlim()[0]), abs(ax.get_xlim()[1]))
        step = 5 if _lim < 25 else 10
        ticks = np.arange(0, _lim + 0.01, step)
        ax.set_xticks(ticks)
        ax.set_xticklabels([f"{int(t)}%" for t in ticks])

    fig.suptitle("Куда исчезают профессиональные разработчики после 30?", color=FG)
    fig.legend(handles=[plt.Rectangle((0,0),1,1, color=GRAY1), plt.Rectangle((0,0),1,1, color=RED)], labels=["Деятельность в области информации и связи", "Профессиональные разработчики"], loc="lower left", bbox_to_anchor=(0.05, 0.09), framealpha=0)
    fig.legend(handles=[plt.Rectangle((0,0),1,1, color=GRAY1), plt.Rectangle((0,0),1,1, color=GRAY2)], labels=["Деятельность в области образования", "Здравоохранение и социальные услуги"], loc="lower right", bbox_to_anchor=(0.95, 0.09), framealpha=0)

    fig.savefig(os.path.join(OUT_DIR, "13_age_is40new60.png"), dpi=300, facecolor=BG)
    plt.close(fig)